In [ ]:
# Cell 1: Imports (Lấy từ file cũ của cậu)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

from bs4 import BeautifulSoup
import time
import json
from datetime import datetime
import re
import pandas as pd
import numpy as np
import random
from urllib.parse import urljoin

import undetected_chromedriver as uc

import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

print("Cell 1: Imports cho Script 2 đã sẵn sàng.")

In [ ]:
# Cell 2: Configuration
load_dotenv()

MAX_URLS_TO_PROCESS_IN_SESSION = 5

# --- Cấu hình Database ---
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_PORT = os.getenv('DB_PORT', '5432')

# Tên các bảng chúng ta sẽ làm việc
DB_URL_QUEUE_TABLE = 'topcv_job_url_queue'       # Bảng hàng đợi (để đọc)
DB_DETAILED_TABLE = 'topcv_jobs_detailed'         # Bảng chi tiết (để ghi)

print("Cell 2: Cấu hình cho Script 2 đã được thiết lập.")
print(f"Xử lý tối đa: {MAX_URLS_TO_PROCESS_IN_SESSION} URLs mỗi phiên.")

In [ ]:
# Cell 3: WebDriver Initialization
driver = None
try:
    print("Cell 3: Đang cấu hình và khởi tạo WebDriver...")
    chrome_options_uc = Options()
    
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    chrome_options_uc.add_argument(f"--user-agent={USER_AGENT}")
    chrome_options_uc.add_argument("--window-size=1920,1080")
    chrome_options_uc.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = uc.Chrome(options=chrome_options_uc)
    driver.set_page_load_timeout(180) 
    
    print("WebDriver đã được khởi tạo và cấu hình thành công!")

except Exception as e:
    print(f"Lỗi khi khởi tạo WebDriver: {e}")
    driver = None

In [ ]:
# Cell 4: Helper Functions - Parse Job Detail Page Content

def get_job_detail_page_source(driver_instance, detail_url, wait_time=30): # Giữ wait_time = 30 giây
    """Điều hướng đến trang chi tiết job và trả về page source."""
    try:
        driver_instance.get(detail_url)
        WebDriverWait(driver_instance, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.job-detail__information-detail--content"))
        )
        time.sleep(random.uniform(2, 4)) 
        return driver_instance.page_source
    except TimeoutException:
        print(f"    Timeout ({wait_time}s) khi chờ element trên trang chi tiết: {detail_url}.")
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S%f") 
            filename_base = f"debug_timeout_{detail_url.split('/')[-1].split('?')[0]}_{timestamp}"
            if driver_instance: # Kiểm tra trước khi dùng
                driver_instance.save_screenshot(f"{filename_base}.png")
                print(f"      Đã lưu screenshot: {filename_base}.png")
                with open(f"{filename_base}.html", "w", encoding="utf-8") as f_html_err:
                    f_html_err.write(driver_instance.page_source)
                print(f"      Đã lưu page source: {filename_base}.html")
        except Exception as e_save_debug:
            print(f"      Không thể lưu thông tin debug khi timeout: {e_save_debug}")
    except Exception as e: 
        print(f"    Lỗi khi truy cập hoặc xử lý trang chi tiết {detail_url}: {e}")
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S%f")
            filename_base = f"debug_error_{detail_url.split('/')[-1].split('?')[0]}_{timestamp}"
            if driver_instance: 
                 driver_instance.save_screenshot(f"{filename_base}.png")
                 print(f"      Đã lưu screenshot khi lỗi: {filename_base}.png")
                 with open(f"{filename_base}.html", "w", encoding="utf-8") as f_html_err:
                     f_html_err.write(driver_instance.page_source)
                 print(f"      Đã lưu page source khi lỗi: {filename_base}.html")
        except Exception as e_save_debug:
            print(f"      Không thể lưu thông tin debug khi lỗi: {e_save_debug}")
    return None

def parse_company_info_from_detail(soup_detail):
     company_info = {'company_name_detail': None, 'company_scale': None, 'company_field': None, 'company_full_address': None}
     container = soup_detail.find("div", class_="job-detail__company--information")
     if not container: return company_info
     if name_tag := container.select_one("div.company-name-label a.name"): company_info['company_name_detail'] = name_tag.get_text(strip=True)
     if scale_item := container.find("div", class_="company-scale"): 
         if val_tag := scale_item.find("div", class_="company-value"): company_info['company_scale'] = val_tag.get_text(strip=True)
     if field_item := container.find("div", class_="company-field"):
         if val_tag := field_item.find("div", class_="company-value"): company_info['company_field'] = val_tag.get_text(strip=True)
     if address_item := container.find("div", class_="company-address"):
         if val_div := address_item.find("div", class_="company-value"):
             company_info['company_full_address'] = val_div.get('data-original-title', val_div.get_text(strip=True)) # Ưu tiên data-original-title
     return company_info

def parse_general_job_info_from_detail(soup_detail):
     general_info = {'job_level': None, 'education_level': None, 'quantity_needed': None, 'employment_type': None, 'gender_requirement': None}
     box = soup_detail.find("div", class_="job-detail__body-right--box-general")
     if not box: return general_info
     for group in box.find_all("div", class_="box-general-group"):
         if title_tag := group.find("div", class_="box-general-group-info-title"):
             if value_tag := group.find("div", class_="box-general-group-info-value"):
                 title, value = title_tag.get_text(strip=True), value_tag.get_text(strip=True)
                 if "Cấp bậc" in title: general_info['job_level'] = value
                 elif "Học vấn" in title: general_info['education_level'] = value
                 elif "Số lượng tuyển" in title: general_info['quantity_needed'] = value 
                 elif "Hình thức làm việc" in title: general_info['employment_type'] = value
                 elif "Giới tính" in title: general_info['gender_requirement'] = value
     return general_info

def parse_skills_categories_from_detail(soup_detail):
     skills_cats = {'related_job_categories': [], 'required_skills_tags': [], 'preferred_skills_tags': []}
     container = soup_detail.find("div", class_="job-detail__body-right--box-category")
     if not container: return skills_cats # Trả về list rỗng nếu không tìm thấy container
     for box in container.find_all("div", class_="box-category"):
         if title_tag := box.find("div", class_="box-title"):
             if tags_container := box.find("div", class_="box-category-tags"):
                 title = title_tag.get_text(strip=True)
                 if "Danh mục Nghề" in title: 
                     skills_cats['related_job_categories'] = [a.get_text(strip=True) for a in tags_container.select("a.box-category-tag")] or []
                 elif "Kỹ năng cần có" in title: 
                     skills_cats['required_skills_tags'] = [s.get_text(strip=True) for s in tags_container.select("span.box-category-tag")] or []
                 elif "Kỹ năng nên có" in title: 
                     skills_cats['preferred_skills_tags'] = [s.get_text(strip=True) for s in tags_container.select("span.box-category-tag")] or []
     return skills_cats

def parse_job_content_from_detail(soup_detail): # CẬP NHẬT PHẦN working_time_text
    content = {'job_description_text': None, 'job_requirements_text': None, 'job_benefits_text': None, 'working_time_text': None}
    content_container = soup_detail.select_one("div.job-detail__information-detail--content div.job-description")
    if not content_container: return content
    
    for item in content_container.find_all("div", class_="job-description__item", recursive=False):
        h3_tag = item.find("h3")
        item_content_div = item.find("div", class_="job-description__item--content")
        
        if h3_tag and item_content_div:
            section_title = h3_tag.get_text(strip=True)
            text_parts = []
            
            if "Thời gian làm việc" in section_title:
                # Xử lý riêng cho "Thời gian làm việc" để lấy các list item
                list_items = item_content_div.find_all("div", class_="job-description__item--content-list")
                if list_items:
                    for li in list_items:
                        if li_text := li.get_text(strip=True):
                            text_parts.append(li_text)
                elif item_content_div.get_text(strip=True): # Fallback nếu không có cấu trúc list
                     text_parts.append(item_content_div.get_text(strip=True))
                section_text = "\n".join(text_parts) if text_parts else None
                content['working_time_text'] = section_text
            else:
                # Xử lý chung cho các mục khác
                for element in item_content_div.children:
                    if isinstance(element, str) and element.strip(): text_parts.append(element.strip())
                    elif element.name == 'p' and element.get_text(strip=True): text_parts.append(element.get_text(strip=True))
                    elif element.name in ['ul', 'ol']:
                        for li_tag in element.find_all('li', recursive=False): 
                            if li_text_content := li_tag.get_text(strip=True): text_parts.append(f"- {li_text_content}")
                section_text = "\n".join(text_parts) if text_parts else None

                if "Mô tả công việc" in section_title: content['job_description_text'] = section_text
                elif "Yêu cầu ứng viên" in section_title: content['job_requirements_text'] = section_text
                elif "Quyền lợi" in section_title: content['job_benefits_text'] = section_text
    return content

def parse_application_deadline_from_detail(soup_detail):
     if deadline_tag := soup_detail.find("div", class_="job-detail__information-detail--actions-label"):
         if match := re.search(r'(\d{2}/\d{2}/\d{4})', deadline_tag.get_text(strip=True)):
             return match.group(1)
     return None

print("Cell 4: Các hàm helper parse trang chi tiết đã được định nghĩa.")

In [ ]:
# Cell 5: Database Schema for Detailed Jobs (PHIÊN BẢN HOÀN CHỈNH)

CREATE_DETAILED_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS {DB_DETAILED_TABLE} (
    job_id SERIAL PRIMARY KEY,
    job_url TEXT UNIQUE,
    job_title TEXT, -- Sẽ được điền dữ liệu
    
    scrape_date DATE,
    inserted_at TIMESTAMP WITHOUT TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    
    -- Thông tin công ty từ parse_company_info_from_detail
    company_name_detail TEXT, 
    company_scale TEXT,
    company_field TEXT,
    company_full_address TEXT,

    -- Thông tin chung từ parse_general_job_info_from_detail
    job_level TEXT,
    education_level TEXT, -- Đổi tên từ education_level_text để khớp với hàm parse
    quantity_needed TEXT, -- Lưu dạng text gốc
    employment_type TEXT,
    gender_requirement TEXT, -- Sẽ được điền dữ liệu
    
    -- Lấy từ các nguồn khác trên trang chi tiết
    salary_text_detail TEXT, 
    experience_text_detail TEXT,
    application_deadline_date TEXT,
    
    -- Các thẻ kỹ năng, danh mục (đã join thành string)
    related_job_categories_text TEXT,
    required_skills_tags_text TEXT,
    preferred_skills_tags_text TEXT,
    
    -- Nội dung chính từ parse_job_content_from_detail
    job_description_text TEXT,
    job_requirements_text TEXT,
    job_benefits_text TEXT,
    working_time_text TEXT -- Sẽ được điền dữ liệu
);
"""

print("Cell 5: Schema cho bảng dữ liệu chi tiết đã được cập nhật và hoàn thiện.")

In [ ]:
# Cell 6: Main Detail Scraping and Storing Logic (PHIÊN BẢN SỬA LỖI TOÀN DIỆN)

if driver and all([DB_HOST, DB_NAME, DB_USER, DB_PASSWORD]):
    conn = None
    cur = None
    jobs_processed_this_session = 0
    jobs_succeeded = 0
    jobs_failed = 0
    
    try:
        print("Đang kết nối đến PostgreSQL...")
        conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
        cur = conn.cursor()
        print("Kết nối thành công!")

        cur.execute(CREATE_DETAILED_TABLE_SQL)
        conn.commit()
        print(f"Bảng '{DB_DETAILED_TABLE}' đã sẵn sàng.")

        get_urls_query = sql.SQL("SELECT job_url FROM {} WHERE status = 'pending' LIMIT %s;").format(
            sql.Identifier(DB_URL_QUEUE_TABLE)
        )
        cur.execute(get_urls_query, (MAX_URLS_TO_PROCESS_IN_SESSION,))
        urls_to_process = [row[0] for row in cur.fetchall()]

        if not urls_to_process:
            print("Không có URL nào ở trạng thái 'pending' để xử lý.")
        else:
            print(f"Bắt đầu xử lý {len(urls_to_process)} URLs...")

            for i, detail_url in enumerate(urls_to_process):
                jobs_processed_this_session += 1
                print(f"\\n--- Xử lý URL {i+1}/{len(urls_to_process)} ---")
                print(f"  URL: {detail_url}")

                try:
                    detail_html = get_job_detail_page_source(driver, detail_url)
                    if not detail_html:
                        raise Exception("Không thể lấy được page source.")

                    soup_detail = BeautifulSoup(detail_html, 'html.parser')

                    # --- BƯỚC 6: THU THẬP VẬT LIỆU (CẨN THẬN HƠN) ---
                    
                    # 6.1. Gọi tất cả các hàm parse của cậu
                    company_info = parse_company_info_from_detail(soup_detail)
                    general_info = parse_general_job_info_from_detail(soup_detail)
                    skills_cats = parse_skills_categories_from_detail(soup_detail)
                    job_content = parse_job_content_from_detail(soup_detail)
                    
                    # 6.2. Lấy các thông tin riêng lẻ một cách tường minh
                    title_tag = soup_detail.select_one("h1.job-detail__title")
                    job_title = title_tag.get_text(strip=True) if title_tag else None

                    salary_tag = soup_detail.select_one("div.job-detail__information--value-salary")
                    salary_text = salary_tag.get_text(strip=True) if salary_tag else None
                    
                    # Sửa selector cho Experience để hết cảnh báo và chính xác hơn
                    exp_value_tag = None
                    for group in soup_detail.find_all("div", class_="box-general-group"):
                        title_tag = group.find("div", class_="box-general-group-info-title")
                        if title_tag and "Kinh nghiệm" in title_tag.get_text():
                            exp_value_tag = group.find("div", class_="box-general-group-info-value")
                            break # Tìm thấy là dừng ngay
                    experience_text = exp_value_tag.get_text(strip=True) if exp_value_tag else None
                    
                    deadline = parse_application_deadline_from_detail(soup_detail)

                    # --- BƯỚC 7: ĐÓNG GÓI VẬT LIỆU (ĐẦY ĐỦ) ---
                    data_to_insert = {
                        # Thông tin cơ bản
                        'job_url': detail_url,
                        'job_title': job_title, # ĐÃ THÊM VÀO
                        'scrape_date': datetime.now().date(),
                        
                        # Unpack từ các hàm parse
                        **company_info,
                        **general_info, # chứa job_level, education_level, quantity_needed, employment_type, gender_requirement
                        **job_content,  # chứa job_description_text, job_requirements_text, job_benefits_text, working_time_text

                        # Các trường riêng lẻ
                        'salary_text_detail': salary_text,
                        'experience_text_detail': experience_text,
                        'application_deadline_date': deadline,
                        
                        # Chuyển list thành string
                        'related_job_categories_text': ', '.join(skills_cats['related_job_categories']) if skills_cats.get('related_job_categories') else None,
                        'required_skills_tags_text': ', '.join(skills_cats['required_skills_tags']) if skills_cats.get('required_skills_tags') else None,
                        'preferred_skills_tags_text': ', '.join(skills_cats['preferred_skills_tags']) if skills_cats.get('preferred_skills_tags') else None
                    }
                    
                    # --- BƯỚC 8: GỬI VỀ KHO ---
                    columns = data_to_insert.keys()
                    # Đảm bảo thứ tự values khớp với columns
                    values = [data_to_insert[col] for col in columns]

                    insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES %s ON CONFLICT (job_url) DO NOTHING").format(
                        sql.Identifier(DB_DETAILED_TABLE),
                        sql.SQL(', ').join(map(sql.Identifier, columns))
                    )
                    cur.execute(insert_stmt, (tuple(values),))

                    # --- BƯỚC 9: CẬP NHẬT HÀNG ĐỢI ---
                    update_queue_query = sql.SQL("UPDATE {} SET status = 'completed', processed_at = CURRENT_TIMESTAMP WHERE job_url = %s;").format(
                        sql.Identifier(DB_URL_QUEUE_TABLE)
                    )
                    cur.execute(update_queue_query, (detail_url,))
                    
                    conn.commit()
                    print(f"  THÀNH CÔNG! Đã lưu chi tiết và cập nhật hàng đợi.")
                    jobs_succeeded += 1

                except Exception as e:
                    conn.rollback()
                    update_queue_query = sql.SQL("UPDATE {} SET status = 'error', processed_at = CURRENT_TIMESTAMP WHERE job_url = %s;").format(
                        sql.Identifier(DB_URL_QUEUE_TABLE)
                    )
                    cur.execute(update_queue_query, (detail_url,))
                    conn.commit()
                    print(f"  !!! LỖI khi xử lý URL. Đã cập nhật trạng thái thành 'error'. Chi tiết: {e}")
                    jobs_failed += 1
                
                time.sleep(random.uniform(7, 15))

    finally:
        print(f"\\n--- KẾT THÚC SCRIPT 2 ---")
        if 'jobs_processed_this_session' in locals():
            print(f"Tổng số URL được giao: {jobs_processed_this_session}")
            print(f"  - Thành công: {jobs_succeeded}")
            print(f"  - Thất bại: {jobs_failed}")
        if cur: cur.close()
        if conn: conn.close(); print("Đã đóng kết nối DB.")
        if driver: driver.quit(); print("Đã đóng WebDriver.")
else:
    print("Script không thể chạy do WebDriver hoặc cấu hình DB bị thiếu.")